# Entregável 1 — Auditoria Analítica e Matriz de Riscos

## 1. As 3 Anomalias Identificadas

### Anomalia 1: Indisponibilidade Crítica de Geração (Março a Junho/2024)
* **Diagnóstico:** Conforme observado no histórico entre março e julho de 2024, a geração mensal da Usina Teste sofreu uma queda abrupta. Em Fevereiro/2024, a usina gerou pouco mais de 300 MWh, despencando para aproximadamente 23 MWh em Março — uma queda brusca de cerca de 92%.
* **Causa Raiz Provável:** Embora a irradiação solar em março seja ligeiramente inferior à de fevereiro, essa variação sazonal não justifica um colapso desta magnitude. O comportamento indica interrupção não programada, como falha crítica em equipamentos centrais (ex.: inversores ou transformador), parada técnica prolongada ou ordem de restrição/corte de geração (*curtailment*) emitida pela distribuidora.
* **Impacto no Negócio:** Atraso ou incapacidade de fornecimento de créditos às UCs do grupo, gerando exposição tarifária involuntária à distribuidora.

---

### Anomalia 2: Desbalanço Estrutural da Carteira de UCs (Déficit Contínuo de Créditos)
* **Diagnóstico:** No balanço mensal comparativo entre a geração total da usina (X) e o consumo acumulado das UCs (Y), observa-se que, a partir de Dezembro de 2023, o consumo passa a superar sistematicamente a geração disponível no mês.
* **Causa Raiz Provável:** Variações pontuais em que Y > X são normais e justificam o acúmulo de créditos (Z). Contudo, quando o saldo permanece insuficiente por vários meses subsequentes, evidencia-se falha de gestão: a carteira de UCs está superdimensionada para a capacidade firme do ativo.
* **Diretriz de Negócio e Ação Corretiva:**
  * **Redespacho / Remanejo de Matriz:** Remanejar parcelas das UCs excedentes para outras usinas com superávit de geração no mesmo pátio/distribuidora.
  * **Adequação do Rateio no Algoritmo de P&D:** Recalcular dinamicamente as cotas percentuais de alocação mensalmente, garantindo que UCs de maior prioridade financeira recebam os créditos disponíveis sem expor o cliente à tarifa cheia.

---

### Anomalia 3: Inconsistência de Dados no Consumo da UC (Y < 0)
* **Diagnóstico:** Identificaram-se registros em que o consumo físico de energia da Unidade Consumidora consta como valor negativo (Y < 0), o que é matematicamente impossível para uma UC consumidora pura.
* **Causa Raiz Provável:**
  * **Ajuste Faturado pela Distribuidora:** Cobrança prévia por estimativa com estorno/ajuste negativo aplicado na leitura presencial seguinte.
  * **Falha no Pipeline de Dados / OCR Parser:** Erro na extração automatizada de faturas em PDF (troca de colunas de débito/crédito ou leitura incorreta de sinais).
* **Tratamento:** Substituir valores Y < 0 por `NaN`, imputar o consumo utilizando a média móvel dos últimos 3 meses válidos e sinalizar o registro com a tag `FLAG_AJUSTE_DISTRIBUIDORA`.

---

## 2. Matriz de Risco

| Anomalia / Inconsistência | Categoria | Severidade | Impacto no Modelo / Negócio | Ação Mapeada / Mitigação |
| :--- | :--- | :---: | :--- | :--- |
| **1. Queda Abrupta na Geração** *(Mar a Jun/2024)* | Operacional / Ativo | **Alta** | Distorce a previsão de capacidade futura da usina, gerando subestimativa no rateio. | Tratar como *outlier* no pipeline de dados e solicitar verificação operacional da usina (O&M). |
| **2. Desbalanço Estrutural de Créditos** *(A partir de Dez/2023)* | Negócio / Regulatório | **Crítica** | Esgota o saldo de créditos (Z), expondo UCs prioritárias ao pagamento de tarifa cheia. | Reordenar cotas dinamicamente no Motor de Rateio e avaliar remanejo de UCs excedentes. |
| **3. Consumo Negativo na UC** *(Y < 0)* | Qualidade de Dados | **Média** | Deturpa médias móveis de consumo e afeta o treinamento do modelo de Machine Learning. | Sanitizar no ETL (substituir por `NaN` + média móvel) e aplicar a flag `FLAG_AJUSTE_DISTRIBUIDORA`. |

---

# Entregável 2 — Tradução Negócio-Tecnologia (Backlog Ágil)

### Objetivo da Sprint
O objetivo do time de P&D é construir o **Motor de Rateio Dinâmico de Créditos**. O sistema precisa analisar a geração da usina (X), o consumo de cada cliente (Y) e o saldo de créditos acumulados (Z) para distribuir essa energia de forma otimizada.

---

### User Story 1: Distribuição Inteligente de Créditos
* **História:** **Como** Gestor do Projeto, **eu quero** um algoritmo que ajuste a divisão de créditos das UCs todo mês conforme o consumo real delas, **para que** nenhum cliente prioritário fique sem crédito e tenha que pagar a conta cara da distribuidora.
* **Critérios de Aceite:**
  * **Trava de consumo zero:** Se uma UC ficar 2 meses sem consumir nada ($Y = 0$), o sistema deve reduzir a entrega de créditos para ela para no máximo 5%, repassando o restante para quem precisa.
  * **Prioridade aos melhores clientes:** Clientes prioritários devem ter pelo menos 90% do seu consumo coberto por créditos no mês.
  * **Alerta de falta de energia:** Se o consumo total for maior do que a usina consegue entregar ($\sum Y > X + \sum Z$), o sistema deve emitir um alerta automático e indicar quais clientes devem ser transferidos para outra usina.

---

### User Story 2: Limpeza e Correção Automática dos Dados
* **História:** **Como** Cientista de Dados, **eu quero** que o sistema identifique e corrija sozinho dados errados (como consumos negativos e apagões da usina), **para que** o modelo de Inteligência Artificial não aprenda com informações incorretas.
* **Critérios de Aceite:**
  * **Consumo negativo ($Y < 0$):** Se aparecer consumo negativo na planilha, o sistema deve apagar esse valor, marcar como "Ajuste da Distribuidora" e preencher com a média dos últimos 3 meses daquele cliente.
  * **Apagão da usina:** Se a geração da usina cair mais de 80% de um mês para o outro sem explicação do tempo, o sistema deve isolar esse mês para não estragar as previsões futuras de geração.
  * **Crédito nunca fica negativo:** O sistema deve travar a conta para que o saldo acumulado de créditos ($Z$) nunca fique abaixo de zero.

---

### User Story 3: Previsão dos Próximos Meses
* **História:** **Como** Analista de Energia, **eu quero** que o modelo preveja o consumo e a geração dos próximos 3 meses, **para que** a gente saiba com antecedência se vai faltar ou sobrar energia no grupo.
* **Critérios de Aceite:**
  * **Acurácia da previsão:** O modelo deve acertar a previsão de consumo com uma margem de erro menor que 8% ($\text{MAPE} < 8\%$).
  * **Aviso de risco:** O sistema deve emitir um relatório avisando quais meses do próximo trimestre terão déficit de geração ($\hat{Y} > \hat{X}$) para podermos agir antes da conta chegar.

---

# Entregável 3 — Relatório Executivo de Gestão

**COMUNICADO INTERNO — DIRETORIA EXECUTIVA DIGITAL GRID**

* **Para:** Diretoria Executiva da Digital Grid
* **De:** Analista de Projetos de P&D (Liderança do Projeto de Rateio)
* **Assunto:** Diagnóstico do Balanço Energético e Diretrizes Estratégicas do Portfólio de GD

---

### Diagnóstico Resumido do Balanço Energético
A análise do histórico operacional da **Usina Teste** em relação ao grupo de Unidades Consumidoras (UCs) vinculadas revelou dois pontos de atenção críticos para a sustentabilidade financeira do arranjo:

1. **Vulnerabilidade Operacional do Ativo:** Entre março e junho de 2024, identificou-se um colapso abrupto de 92% na geração da usina (passando de >300 MWh em Fev/24 para ≈23 MWh em Mar/24). A magnitude dessa queda descarta causas exclusivamente meteorológicas, apontando para uma indisponibilidade severa não programada (falha crítica de inversores/transformador ou corte de geração pela distribuidora).
2. **Desbalanço Estrutural de Carteira:** A partir de Dezembro de 2023, o somatório de consumo das UCs passou a superar sistematicamente a geração mensal da usina ($Y > X$). Embora o uso do saldo de créditos acumulados ($Z$) mitigue déficits sazonais temporários, a persistência desse cenário drenou o estoque de créditos do grupo, caracterizando um superdimensionamento da carteira de clientes para a capacidade firme deste ativo.

---

### Recomendação de Ação de Negócios (Próximo Trimestre)
Para mitigar os riscos de exposição tarifária dos clientes e otimizar a receita do ativo, recomendamos a execução de três ações imediatas no próximo trimestre:

1. **Redespacho e Remanejo de Carteira:** Implementar o remanejo imediato de parcelas das UCs excedentes para outras usinas do portfólio Digital Grid que apresentem superávit de geração no mesmo pátio/distribuidora, reequilibrando a curva de carga.
2. **Desenvolvimento do Motor de Rateio Dinâmico (P&D):** Priorizar na próxima Sprint o algoritmo de alocação flexível de créditos, substituindo percentuais estáticos por cotas calculadas mensalmente em função da prioridade financeira e do histórico de consumo real de cada UC.
3. **Plano de Manutenção Preventiva e SLA:** Estabelecer protocolo de auditoria técnica telemetrada junto ao time de O&M (Operação e Manutenção) para garantir que paralisações de ativos sejam identificadas e contornadas em menos de 24 horas, evitando perdas massivas de geração como a observada no primeiro semestre de 2024.